In [ ]:
%run 10_MNESIS_polychronous-chains.ipynb

In [ ]:
opt = Params()
opt

In [ ]:
opt = Params()
hd = HD_SNN(opt)
hd.net.to(hd.opt.device)
model_filename = Path(data_cache) / f"{hd.opt.datetag}.pth"
model_state_dict = torch.load(model_filename, map_location=torch.device(hd.opt.device))
hd.net.load_state_dict(model_state_dict)
hd.net.eval()
print(f"Model weights loaded from {model_filename}")

### testing inference - initiating with a fraction of the neurons

In [ ]:
with torch.no_grad():
    target_full = torch.nn.functional.pad(hd.target, (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay)
    input_spikes[:, :(opt.N_neuron//8), (hd.opt.N_pretime):(hd.opt.N_pretime+hd.opt.num_delay)] *= 0
    _, _, spikes = hd.forward_pass(input_spikes)
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
    target_evoked = hd.target[:, :, hd.opt.num_delay:]

    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
fig,ax = plt.subplots(figsize=(13, 8))
splt.raster(target_full[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
if figpath is not None: printfig(fig, 'fraction_target_init', fig_width=opt.fig_width, fig_height=opt.fig_width/phi)
spikes.shape, spikes[i_SM, :, :].sum().item(), hd.target[i_SM, :, :].sum().item()

In [ ]:
fig,ax = plt.subplots(figsize=(13, 8))
splt.raster(spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
if figpath is not None: printfig(fig, 'fraction_target', fig_width=opt.fig_width, fig_height=opt.fig_width/phi)
spikes.shape, spikes[i_SM, :, :].sum().item(), hd.target[i_SM, :, :].sum().item()

In [ ]:
def get_scores_fraction(N_scan=opt.N_scan):
    N_neurons = np.linspace(0, hd.opt.N_neuron, N_scan, endpoint=True, dtype=int)
    scores_np = np.zeros(N_scan)
    for i_N_neuron, N_neuron in enumerate(N_neurons):
        with torch.no_grad():
            input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay)
            input_spikes[:, :(N_neuron), (hd.opt.N_pretime):(hd.opt.N_pretime+hd.opt.num_delay)] *= 0

            _, _, spikes = hd.forward_pass(input_spikes)
            spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
            target_evoked = hd.target[:, :, hd.opt.num_delay:]

            scores_np[i_N_neuron] = get_f1score(spikes_evoked, target_evoked)

    return N_neurons, scores_np

npy_filename = Path(data_cache) / f"{hd.opt.datetag}_N_neurons.npz"
lock_filename = Path(data_cache) / f"{hd.opt.datetag}_N_neurons.lock"
if RECOMPUTE:
# if True:
    npy_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    data = np.load(npy_filename)
    N_neurons = data['N_neurons']
    scores_np = data['scores_np']
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Score weights loaded from {model_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Score file not found: {model_filename}, evaluating athe score.")
        lock_filename.touch(exist_ok=True)
        ##################
        N_neurons, scores_np = get_scores_fraction()
        ##################        
        np.savez(npy_filename, N_neurons=N_neurons, scores_np=scores_np)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Score file is locked: {lock_filename}, passing.")


In [ ]:
N_neurons.shape, scores_np.shape

In [ ]:
fig,ax = plt.subplots(figsize=(opt.fig_width, opt.fig_width/phi))
ax.hlines([0, 1], N_neurons[0], N_neurons[-1], 'orange', ls='--')
ax.plot(N_neurons, scores_np)
ax.set_xlabel("Inactive trigger neurons")
ax.set_ylabel("F1-score")

if figpath is not None: 
    printfig(fig, 'fraction_target_score', fig_width=opt.fig_width, fig_height=opt.fig_width/phi)